In [ ]:
%pip install -q openai

In [ ]:
import os
from getpass import getpass

from openai import OpenAI
import textwrap

# API Key Setup - uses environment variable or prompts for input
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

# Client and Model Configuration
client = OpenAI()
MODEL = "gpt-5.2"

## Basic Prompt: Starting Simple

Let's start with a straightforward prompt that asks the model to generate a product description from an image. This establishes our baseline before we apply prompt engineering improvements.

In [ ]:
text_wrapper = textwrap.TextWrapper(width=80)
prompt = '''Act as a fashion retailer, you are responsible for writing effective product descriptions.
Use all of the images to create a product description for the following product.'''

In [ ]:
response = client.responses.create(
    model=MODEL,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {
                    "type": "input_image",
                    "image_url": "https://storage.googleapis.com/strapi_cms_assets/skinny-trousers.webp"
                },
            ],
        }
    ],
    max_output_tokens=300,
)

## What We're Improving

The basic prompt above works, but it has several limitations:

1. **No examples provided** - The model has no reference for the desired style, tone, or length
2. **No length constraints** - Output can be unpredictably long or short
3. **No product context** - When images contain multiple items, the model may describe the wrong one

In the next section, we'll address each of these issues using key prompt engineering techniques:

- **In-context learning**: Providing examples to guide the model's output style
- **Explicit constraints**: Setting clear rules for output length (2-4 sentences)
- **Product context**: Adding metadata to disambiguate which item to describe

## Improved Prompt

The improved prompt adds:
1. Three example descriptions to establish the desired style and tone
2. Explicit rules constraining the output to 2-4 sentences
3. A `product_context` field (empty for now, used in the next section)

In [24]:
examples = [
    "Elevate your professional wardrobe with our Classic Navy Dress Pants, a sartorial choice for the modern gentleman.",
    "Refine your evening attire with our Elegant Black Cocktail Dress, an essential selection for the contemporary woman.",
    "Upgrade your casual ensemble with our Sleek White Sneakers, a versatile addition for the fashion-forward individual."
]
product_context = ""

In [49]:
improved_prompt = f"""Act as a fashion retailer, you are responsible for writing effective
product descriptions. Use all of the images to create a product description for the following
product.

Here are some examples in terms of the style, tone and length for future product descriptions:
1. {examples[0]}
2. {examples[1]}
3. {examples[2]}

Rules:
- The product description must be between 2 - 4 sentences in length.
- The product description must be written in a professional tone.

{product_context}

Use the image(s) to create a product description for the following product"""

In [ ]:
response = client.responses.create(
    model=MODEL,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": improved_prompt},
                {
                    "type": "input_image",
                    "image_url": "https://storage.googleapis.com/strapi_cms_assets/skinny-trousers.webp"
                },
            ],
        }
    ],
    max_output_tokens=300,
)

## Multiple Images and Product Context

Product photos often include multiple items. Use `product_context` to specify which item to describe.

In this example:
- We pass multiple images showing the product from different angles
- We explicitly state which item to describe: "Champion cuffed cargo trousers in black"
- We clarify what to ignore: "It is not the hoodie"

In [53]:
product_context = "The item is Champion cuffed cargo trousers in black. It is not the hoodie."

improved_prompt = f"""Act as a fashion retailer, you are responsible for writing effective
product descriptions. Use all of the images to create a product description for the following
product.

Here are some examples in terms of the style, tone and length for future product descriptions:
1. {examples[0]}
2. {examples[1]}
3. {examples[2]}

Rules:
- The product description must be between 2 - 4 sentences in length.
- The product description must be written in a professional tone.

{product_context}

Use the image(s) to create a product description for the following product"""

In [ ]:
response = client.responses.create(
    model=MODEL,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": improved_prompt},
                {
                    "type": "input_image",
                    "image_url": "https://storage.googleapis.com/strapi_cms_assets/205878603-2.webp"
                },
                {
                    "type": "input_image",
                    "image_url": "https://storage.googleapis.com/strapi_cms_assets/205878603-3.jpeg"
                },
            ],
        }
    ],
    max_output_tokens=300,
)

## Your Turn

Now it's your turn to practice writing product descriptions with vision models!

### Exercise: Create Your Own Product Description Prompt

1. **Choose a different product category** - Try electronics, home decor, sports equipment, or beauty products instead of fashion

2. **Write new examples** - Create 3 example product descriptions that match your chosen category's tone and style:

```python
# Example for electronics:
examples = [
    "Experience seamless connectivity with our Premium Wireless Earbuds, engineered for the audiophile who demands exceptional sound quality.",
    "Transform your workspace with our Ultra-Wide Curved Monitor, designed for professionals who value both productivity and visual excellence.",
    "Power through your day with our Compact Power Bank, the essential companion for the always-connected traveler."
]
```

3. **Customize the prompt** - Modify the `improved_prompt` template to fit your product category:
   - Update the role (e.g., "Act as an electronics retailer...")
   - Adjust the rules for your category's conventions
   - Add relevant product context

4. **Test with real images** - Find product images online and test your prompt:

```python
# Your custom prompt here
your_prompt = f"""Act as a [YOUR CATEGORY] retailer...

Examples:
1. {examples[0]}
2. {examples[1]}
3. {examples[2]}

Rules:
- [Your rules here]

Use the image(s) to create a product description for the following product"""

# Test your prompt
response = client.responses.create(
    model=MODEL,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": your_prompt},
                {
                    "type": "input_image",
                    "image_url": "YOUR_IMAGE_URL_HERE"
                },
            ],
        }
    ],
    max_output_tokens=300,
)
print(response.output_text)
```

### Challenge

Try generating descriptions for the same product with different example styles. How does changing the examples affect the output? Experiment with:
- Formal vs. casual tone
- Short (1 sentence) vs. longer (3-4 sentences) examples
- Different adjectives and vocabulary